# Aula 4 | Busca Sem Informação em Python
### Inteligência Artificial (PCO118A) — RA1
**Professor:** Carlos Costa Cedro

Neste laboratório vocês vão implementar, com as próprias mãos, as três estratégias de busca sem informação estudadas na aula: **busca em largura**, **busca em profundidade** e **busca de custo uniforme**.

O objetivo não é apenas fazer o código funcionar. É comprovar experimentalmente três afirmações feitas na aula teórica:

1. Largura e profundidade são quase o mesmo algoritmo. A diferença cabe em uma única linha.
2. Nenhuma das duas é universalmente melhor. Depende de onde o objetivo está.
3. Quando os passos têm custos diferentes, o caminho com menos passos deixa de ser o mais barato.

**Como trabalhar:** em duplas. Executem as células na ordem. Onde houver `# TODO`, o código está incompleto e precisa ser escrito por vocês.


---
## Atividade 1: Uma linha muda tudo

Vamos usar o mesmo grafo do Duelo de Estratégias feito no papel.

```
A
├── B
│   ├── D
│   │   ├── F
│   │   │   ├── H
│   │   │   │   ├── J
│   │   │   │   └── K
│   │   │   └── I
│   │   └── G
│   │       ├── P
│   │       └── Q
│   └── E
│       ├── N
│       └── O
└── C
    ├── L
    └── M
```

Execute a célula abaixo para carregar o grafo.


In [ ]:
GRAFO = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['L', 'M'],
    'D': ['F', 'G'],
    'E': ['N', 'O'],
    'F': ['H', 'I'],
    'G': ['P', 'Q'],
    'H': ['J', 'K'],
    'I': [], 'J': [], 'K': [], 'L': [], 'M': [], 'N': [], 'O': [], 'P': [], 'Q': []
}

print("Grafo carregado com", len(GRAFO), "nos.")

### 1.1 Implementando a busca genérica

A função abaixo implementa **as duas estratégias ao mesmo tempo**. Ela mantém duas listas:

- `open_`: caminhos descobertos e ainda não examinados, ou seja, a **fronteira**
- `closed`: nós já examinados

A única diferença entre largura e profundidade está em **onde os novos caminhos entram em `open_`**:

- **Largura:** novos caminhos entram no **fim** da lista, que funciona como fila FIFO
- **Profundidade:** novos caminhos entram no **início** da lista, que funciona como pilha LIFO

Complete a linha marcada com `# TODO`.


In [ ]:
def busca_generica(grafo, inicio, objetivo, estrategia):
    """Executa busca em largura ou em profundidade sobre um grafo.

    estrategia: 'largura' ou 'profundidade'
    Retorna um dicionario com caminho, nos examinados e tamanho maximo da fronteira.
    """
    open_ = [[inicio]]          # fronteira: lista de caminhos
    closed = set()              # nos ja examinados
    examinados = 0
    max_fronteira = 1
    ordem_exame = []

    while open_:
        caminho = open_.pop(0)  # sempre retira o primeiro
        no = caminho[-1]
        examinados += 1
        ordem_exame.append(no)

        if no == objetivo:
            return {
                'caminho': caminho,
                'passos': len(caminho) - 1,
                'examinados': examinados,
                'max_fronteira': max_fronteira,
                'ordem_exame': ordem_exame
            }

        closed.add(no)

        novos = []
        for vizinho in grafo[no]:
            ja_visto = vizinho in closed or any(vizinho == c[-1] for c in open_)
            if not ja_visto:
                novos.append(caminho + [vizinho])

        # TODO: complete as duas linhas abaixo
        # Se a estrategia for 'largura', os novos caminhos vao para o FIM de open_
        # Se for 'profundidade', vao para o INICIO de open_
        if estrategia == 'largura':
            open_ = None   # <-- substitua
        else:
            open_ = None   # <-- substitua

        max_fronteira = max(max_fronteira, len(open_))

    return None

### 1.2 Rodada 1: o objetivo é o nó **L**

Execute e anote os resultados.


In [ ]:
for estrategia in ['largura', 'profundidade']:
    r = busca_generica(GRAFO, 'A', 'L', estrategia)
    print(f"{estrategia.upper():14s} | examinados: {r['examinados']:2d} | "
          f"passos: {r['passos']} | fronteira max: {r['max_fronteira']}")
    print(f"{'':14s} | caminho: {' -> '.join(r['caminho'])}")
    print(f"{'':14s} | ordem:   {' '.join(r['ordem_exame'])}")
    print()

### 1.3 Rodada 2: o objetivo passa a ser o nó **J**

Antes de executar, **façam uma aposta**: qual estratégia vai examinar menos nós desta vez?


In [ ]:
for estrategia in ['largura', 'profundidade']:
    r = busca_generica(GRAFO, 'A', 'J', estrategia)
    print(f"{estrategia.upper():14s} | examinados: {r['examinados']:2d} | "
          f"passos: {r['passos']} | fronteira max: {r['max_fronteira']}")
    print(f"{'':14s} | caminho: {' -> '.join(r['caminho'])}")
    print(f"{'':14s} | ordem:   {' '.join(r['ordem_exame'])}")
    print()

### 1.4 Perguntas da Atividade 1

Respondam aqui, na própria célula de texto:

**a)** Quantos nós cada estratégia examinou em cada rodada? O que aconteceu com os números entre a Rodada 1 e a Rodada 2?

**b)** Quantas linhas de código diferem entre a busca em largura e a busca em profundidade?

**c)** Se vocês não soubessem onde o objetivo está no grafo, qual estratégia escolheriam? Justifiquem.

*Escrevam as respostas abaixo:*

---



---
## Atividade 2: Aplicando ao problema dos baldes

Vamos reutilizar **exatamente a mesma função** `busca_generica` em um problema completamente diferente: o problema dos baldes da Aula 3.

Um balde de 4 litros e outro de 3 litros, sem marcações. Objetivo: obter 2 litros no balde maior.

Estado: par ordenado `(x, y)`, com `x` no balde de 4 e `y` no balde de 3.


In [ ]:
def sucessores(estado):
    """Retorna todos os estados alcancaveis a partir de estado com um unico operador."""
    x, y = estado
    CAP_A, CAP_B = 4, 3

    resultados = []
    resultados.append((CAP_A, y))                  # encher A
    resultados.append((x, CAP_B))                  # encher B
    resultados.append((0, y))                      # esvaziar A
    resultados.append((x, 0))                      # esvaziar B

    # TODO: implemente os dois operadores de transferencia
    # Dica: a quantidade transferida e o MINIMO entre o que existe na origem
    #       e o espaco livre no destino.
    transf = None                                  # <-- de A para B
    resultados.append((x - transf, y + transf))

    transf = None                                  # <-- de B para A
    resultados.append((x + transf, y - transf))

    return resultados


# teste rapido: a partir de (0,0) quais estados sao alcancaveis?
print(sucessores((0, 0)))

### 2.1 Adaptando a busca para o problema dos baldes

O problema dos baldes não tem um grafo pronto em um dicionário. Os vizinhos são **calculados** pela função `sucessores`. Além disso, o objetivo não é um estado único, é qualquer estado em que o balde maior tenha 2 litros.


In [ ]:
def busca_baldes(estrategia):
    inicio = (0, 0)
    open_ = [[inicio]]
    closed = set()
    examinados = 0
    max_fronteira = 1

    while open_:
        caminho = open_.pop(0)
        estado = caminho[-1]
        examinados += 1

        if estado[0] == 2:                     # teste de objetivo: familia de estados
            return {
                'caminho': caminho,
                'passos': len(caminho) - 1,
                'examinados': examinados,
                'max_fronteira': max_fronteira
            }

        closed.add(estado)

        novos = []
        for s in sucessores(estado):
            if s != estado and s not in closed and not any(s == c[-1] for c in open_):
                novos.append(caminho + [s])

        open_ = open_ + novos if estrategia == 'largura' else novos + open_
        max_fronteira = max(max_fronteira, len(open_))

    return None


for estrategia in ['largura', 'profundidade']:
    r = busca_baldes(estrategia)
    print(f"{estrategia.upper():14s} | examinados: {r['examinados']:2d} | "
          f"passos: {r['passos']} | fronteira max: {r['max_fronteira']}")
    print(f"{'':14s} | caminho: {r['caminho']}")
    print()

### 2.2 Perguntas da Atividade 2

**a)** As duas estratégias encontraram caminhos com o mesmo número de passos? Isso era esperado pela teoria?

**b)** Olhem o valor de `fronteira max`. Ele é muito menor do que na Atividade 1. O que isso diz sobre a **forma** do espaço de estados deste problema?

**c)** A busca em profundidade normalmente corre o risco de se perder em ramos infrutíferos. Por que isso não aconteceu aqui?

*Escrevam as respostas abaixo:*

---



### 2.3 Desafio extra

Alterem as capacidades dos baldes na função `sucessores` e o alvo na função `busca_baldes`. Testem, por exemplo, capacidades 9 e 4 com alvo 6, ou 12 e 7 com alvo 6.

O comportamento relativo entre as duas estratégias se mantém? Por quê?


In [ ]:
# Espaco livre para o desafio extra


---
## Atividade 3: Quando menos passos não significa mais barato

Até agora, todos os passos custaram a mesma coisa. Agora cada trecho tem um custo diferente, como em um mapa rodoviário.

```
        A
      /   \
   2 /     \ 5
    /       \
   B         C
   |          \
 3 |           \ 6
   |            \
   D ---- 2 ---- E
```

Queremos ir de **A** até **E**.


In [ ]:
MAPA = {
    'A': [('B', 2), ('C', 5)],
    'B': [('D', 3)],
    'C': [('E', 6)],
    'D': [('E', 2)],
    'E': []
}

def custo_do_caminho(caminho):
    total = 0
    for i in range(len(caminho) - 1):
        for vizinho, custo in MAPA[caminho[i]]:
            if vizinho == caminho[i + 1]:
                total += custo
    return total

print("Mapa carregado.")

### 3.1 Primeiro, a busca em largura

Ela ignora os custos e procura o caminho com **menos trechos**.


In [ ]:
def largura_mapa(inicio, objetivo):
    open_ = [[inicio]]
    closed = set()
    examinados = 0

    while open_:
        caminho = open_.pop(0)
        no = caminho[-1]
        examinados += 1

        if no == objetivo:
            return caminho, len(caminho) - 1, custo_do_caminho(caminho), examinados

        closed.add(no)
        for vizinho, _ in MAPA[no]:
            if vizinho not in closed and not any(vizinho == c[-1] for c in open_):
                open_.append(caminho + [vizinho])

    return None


caminho, passos, custo, ex = largura_mapa('A', 'E')
print(f"LARGURA        | caminho: {' -> '.join(caminho)}")
print(f"               | passos: {passos} | custo total: {custo} | examinados: {ex}")

### 3.2 Agora, a busca de custo uniforme

Em vez de retirar sempre o primeiro caminho da fronteira, ela retira sempre o **mais barato até o momento**. Para isso usamos uma fila de prioridade, que em Python vem pronta no módulo `heapq`.

Complete a linha marcada com `# TODO`.


In [ ]:
import heapq

def custo_uniforme(inicio, objetivo):
    # cada item da fila e uma tupla (custo_acumulado, caminho)
    fila = [(0, [inicio])]
    closed = set()
    examinados = 0

    while fila:
        custo, caminho = heapq.heappop(fila)   # retira o MAIS BARATO
        no = caminho[-1]
        examinados += 1

        if no == objetivo:
            return caminho, len(caminho) - 1, custo, examinados

        if no in closed:
            continue
        closed.add(no)

        for vizinho, custo_trecho in MAPA[no]:
            if vizinho not in closed:
                # TODO: empilhe o novo caminho na fila com o custo acumulado correto
                heapq.heappush(fila, (None, None))   # <-- substitua

    return None


caminho, passos, custo, ex = custo_uniforme('A', 'E')
print(f"CUSTO UNIFORME | caminho: {' -> '.join(caminho)}")
print(f"               | passos: {passos} | custo total: {custo} | examinados: {ex}")

### 3.3 Perguntas da Atividade 3

**a)** As duas estratégias encontraram o mesmo caminho? Qual tem menos trechos? Qual é mais barato?

**b)** Em que situação a busca em largura e a busca de custo uniforme dariam exatamente o mesmo resultado?

**c)** Se todos os trechos deste mapa custassem 1, qual seria o caminho encontrado pelo custo uniforme?

*Escrevam as respostas abaixo:*

---



---
## Fechamento

Ao final deste laboratório, vocês implementaram três estratégias de busca e comprovaram experimentalmente que:

- A diferença estrutural entre largura e profundidade é mínima, mas o comportamento é radicalmente diferente
- Nenhuma das duas é sempre melhor, o desempenho depende de onde o objetivo está
- Quando os custos variam, é preciso trocar a fila simples por uma fila de prioridade

Na próxima aula veremos como usar **informação sobre o problema** para escolher melhor qual nó da fronteira expandir. É o começo da busca heurística.
